# 💘 ¿Habrá Match? - Predictor de Compatibilidad en Speed Dating

## Speed Dating Experiment - Columbia University (2002-2004)

Este proyecto utiliza datos del experimento de **speed dating** conducido por la Universidad de Columbia entre 2002 y 2004. El experimento consistió en **21 oleadas (waves)** de eventos donde participantes de posgrado tuvieron aproximadamente **10-20 mini-citas de 4 minutos** cada una.

Durante cada cita, los participantes registraron:
- Percepciones mutuas sobre atractivo, sinceridad, inteligencia, diversión, ambición e intereses compartidos
- Si deseaban o no una segunda cita con su pareja
- Información sociodemográfica y preferencias personales

Al final del evento, se determinó si hubo **match mutuo** (ambos dijeron sí). El objetivo de este proyecto es predecir la probabilidad de match usando Machine Learning.

## FASE 1 - ENTENDIMIENTO DEL NEGOCIO

### 1.1 Descripción del negocio
Apps de citas, plataformas de matchmaking y eventos de speed dating necesitan maximizar la tasa de matches para retener usuarios. Predecir compatibilidad antes o durante el evento permite mejorar la experiencia, sugerir parejas con mayor probabilidad de éxito y optimizar la organización de los eventos.

### 1.2 Descripción del problema
De cada par de personas que se conocen en un speed dating, ¿habrá match mutuo? El match ocurre cuando ambas personas marcan "sí" al otro. La tasa natural de match es baja (~16%), lo que genera desbalance de clases.

### 1.3 Objetivos de minería
- Desarrollar un modelo predictivo tipo clasificación binaria para predecir si habrá match (1) o no (0) entre dos personas, usando sus perfiles y percepciones mutuas durante la cita.
- Identificar qué atributos tienen mayor peso en la decisión de match.

### 1.4 Tabla de diseño de solución
| Problema | Tipo Minería | Tipo Análisis | Tipo Aprendizaje | Requerimiento | Métodos | Evaluación |
|---|---|---|---|---|---|---|
| Predecir match mutuo | Predictivo | Clasificación binaria | Supervisado | Histórico, variable objetivo | DT, MLP, SVM, KNN, RF, XGB | Acc, Prec, Rec, F1, ROC-AUC |

### 1.5 Evaluación esperada
- Línea base: Accuracy = 84% (prediciendo siempre 0)
- Meta real: ROC-AUC > 0.75

### 1.6 Recursos
- Hardware: CPU estándar
- Software: Python 3.10, scikit-learn, xgboost, pandas, numpy

## FASE 2 - ENTENDIMIENTO DE LOS DATOS

### 2.1 Carga inicial y exploración


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/Speed Dating Data.csv', encoding='latin-1')
print(f'Shape original: {df.shape}')
df.head()

### 2.2 Ciclo de los datos
Los datos fueron generados mediante encuestas en papel durante eventos de speed dating (2002-2004) en Columbia University. El proceso involucra 3 momentos de captura: signup/Time1 (antes), scorecard (durante), y followup (después).

### 2.3 Diccionario de datos


In [ ]:
pd.DataFrame({
    'Variable': ['match', 'gender', 'age', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar', 'like', 'prob'],
    'Descripcion': ['Objetivo', 'Género', 'Edad', 'Atractivo', 'Sinceridad', 'Inteligencia', 'Diversión', 'Ambición', 'Intereses', 'Like', 'Probabilidad'],
    'Tipo': ['Binaria']*11
})

## FASE 3 - PREPARACIÓN DE DATOS

### 3.0 ELIMINAR COLUMNAS IDENTIFICADORAS (PASO OBLIGATORIO)


In [ ]:
cols_identificadoras = ['iid', 'id', 'idg', 'partner', 'pid', 'wave', 'round', 'position', 
                        'positin1', 'order', 'condtn', 'undergrd', 'zipcode', 'career', 'from', 'field']
df.drop(columns=[c for c in cols_identificadoras if c in df.columns], inplace=True)
print(f'Shape después de eliminar IDs: {df.shape}')

In [ ]:
variables_modelo = ['gender', 'age', 'age_o', 'race', 'race_o', 'samerace', 'imprace', 'goal', 
                   'attr', 'sinc', 'intel', 'fun', 'amb', 'shar', 'like', 'prob', 'match']
variables_existentes = [v for v in variables_modelo if v in df.columns]
df = df[variables_existentes].copy()
print(f'Variables seleccionadas: {len(variables_existentes)}')

In [ ]:
import matplotlib.pyplot as plt
conteo = df['match'].value_counts()
plt.figure(figsize=(8, 4))
plt.bar(['No Match (0)', 'Match (1)'], conteo.values, color=['#FF6B6B', '#FF1493'])
plt.title('Distribución de la Variable Objetivo: Match')
plt.ylabel('Cantidad')
for i, v in enumerate(conteo.values):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.show()

In [ ]:
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = df.drop(columns=['match'])
y = df['match']

# Contar clases
n_minoria = y.value_counts()[1]
n_mayoria = y.value_counts()[0]
ratio = n_minoria / n_mayoria
print(f'Ratio original: {ratio:.3f}')

# SMOTE: subir la minoría solo un 25% de su tamaño original (no hasta balancear)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

n_minoria_train = sum(y_train == 1)
n_mayoria_train = sum(y_train == 0)
print(f'Minoria en train: {n_minoria_train}, Mayoría: {n_mayoria_train}')

# Subir minoria solo un 25%
n_minoria_nueva = int(n_minoria_train * 1.25)
if n_minoria_nueva > n_mayoria_train:
    n_minoria_nueva = int(n_mayoria_train * 0.9)  # Maximo 90% de la mayoria
ratio_objetivo = n_minoria_nueva / n_mayoria_train
print(f'Nueva minoría objetivo: {n_minoria_nueva}')
print(f'Ratio SMOTE: {ratio_objetivo:.4f}')

smote = SMOTE(random_state=42, sampling_strategy=ratio_objetivo)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Train balanceado: {X_train_bal.shape}')
print(f'Distribución: {pd.Series(y_train_bal).value_counts().to_dict()}')

In [ ]:
X_train_bal.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
pd.Series(y_train_bal, name='match').to_csv('../data/y_train.csv', index=False)
pd.Series(y_test, name='match').to_csv('../data/y_test.csv', index=False)
print('Datos guardados en data/')